# 第4章　PyTorchの基礎

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 4.1　テンソル ― PyTorchの基本単位

In [ ]:
import torch

# テンソルを作るいくつかの方法
x   = torch.tensor([1.0, 2.0, 3.0])   # Pythonのリストから
z   = torch.zeros(1, 1, 512, 512)     # すべて0（バッチ,チャネル,高さ,幅）
o   = torch.ones(4, 3)                 # すべて1
r   = torch.randn(8, 3, 224, 224)      # 正規乱数（学習の初期値やダミー入力に）
lin = torch.arange(0, 10)              # 0〜9の連番

print(z.shape)   # torch.Size([1, 1, 512, 512])  ← 形（最重要）
print(r.dtype)   # torch.float32                  ← データ型
print(r.device)  # cpu                            ← どこにあるか（CPU/GPU）

In [ ]:
img = torch.randn(1, 512, 512)          # (C,H,W) の1枚

# ① バッチ次元を足す（1枚をモデルに入れるとき必須）
batch = img.unsqueeze(0)                 # (1, 1, 512, 512)

# ② 形を組み替える（全結合層に渡す前の「平坦化」など）
flat = batch.reshape(1, -1)              # (1, 262144)  -1は「残りを自動計算」

# ③ 軸の順番を入れ替える（HWC ↔ CHW の変換など）
hwc = torch.randn(224, 224, 3)           # 画像ライブラリはHWC（高さ,幅,色）で返しがち
chw = hwc.permute(2, 0, 1)               # (3, 224, 224) PyTorchが期待するCHWへ

# ④ GPUへ送る / NumPyと相互変換
device = "cuda" if torch.cuda.is_available() else "cpu"
img_gpu = img.to(device)                 # テンソルをGPUに載せる
np_array = img.cpu().numpy()             # NumPy配列へ（CPU上でのみ可能）
back     = torch.from_numpy(np_array)    # NumPy配列からテンソルへ

In [ ]:
volume = torch.randn(1, 1, 64, 128, 128)     # 3Dの模擬データ（ブロードキャストの説明用）
mean, std = volume.mean(), volume.std()       # 全体の平均・標準偏差（スカラー）
normalized = (volume - mean) / std            # 形が違っても自動で引き伸ばして計算（正規化）

## 手を動かす ― 形を印字しながら、ブロードキャストの罠を踏む

In [ ]:
import torch

x = torch.randn(8, 3, 224, 224)       # (B, C, H, W) の学習バッチ
print(x.shape)                         # torch.Size([8, 3, 224, 224])
print(x.mean(dim=(0, 2, 3)).shape)     # チャネルごとの平均 → torch.Size([3])

# チャネルごとに違う平均を引きたい。だが形を間違えると…
mean = torch.tensor([0.5, 0.4, 0.3])   # (3,) ← RGB各チャネルの平均のつもり
try:
    y = x - mean                        # 末尾どうし 224 と 3 が合わずエラー
except RuntimeError as e:
    print("失敗:", str(e)[:45])

# 正しくは (3,) を (1, 3, 1, 1) に整形し、「C軸」に合わせる
mean = mean.view(1, 3, 1, 1)
y = x - mean                            # ここで初めてブロードキャストが成立
print(y.shape)                          # torch.Size([8, 3, 224, 224])

## 数字で追う ― permuteの後に潜む「メモリの連続性」（上級・読み飛ばし可）

In [ ]:
import torch

x = torch.arange(24).reshape(2, 3, 4)   # (B, C, W) のつもり
print(x.is_contiguous(), x.stride())     # True  (12, 4, 1)

y = x.permute(0, 2, 1)                    # 軸を入れ替え → (2, 4, 3)
print(y.shape, y.is_contiguous())         # torch.Size([2, 4, 3])  False
print(y.stride())                         # (12, 1, 4) ← 数値は動かず「歩幅」だけ変わった

In [ ]:
try:
    y.view(2, 12)                         # この shape と stride では (2, 12) に view できない
except RuntimeError as e:
    print("失敗:", str(e)[:38])            # 失敗: view size is not compatible with ...

z = y.contiguous().view(2, 12)            # 一度メモリを並べ直せば通る
print(z.shape)                            # torch.Size([2, 12])

## 4.2　自動微分（autograd）― 学習の心臓部

In [ ]:
x = torch.tensor(2.0, requires_grad=True)   # 微分を追跡する
y = x ** 2 + 3 * x + 1                        # 順伝播（計算しながらグラフを記録）
y.backward()                                  # 逆伝播（勾配を計算）
print(x.grad)                                 # tensor(7.)  ← dy/dx = 2x + 3 = 7

In [ ]:
model.eval()
with torch.no_grad():                         # ② 勾配を追跡しない（推論・評価）
    logits = model(image.to(device))
    prob = logits.softmax(dim=1)              # クラス確率へ
    score = prob.max().item()                 # ③ Pythonのfloatとして取り出す

## 手を動かす ― 「勾配は足し込まれる」を数値で目撃する

In [ ]:
import torch

w = torch.tensor(1.0, requires_grad=True)

# リセットせずに2回 backward してみる
for step in range(2):
    loss = (w * 3) ** 2
    loss.backward()
    print(f"step {step}: w.grad = {w.grad.item()}")
# step 0: w.grad = 18.0
# step 1: w.grad = 36.0   ← ！ 前回の18に足し込まれて2倍になった

In [ ]:
for step in range(2):
    if w.grad is not None:
        w.grad.zero_()          # optimizer.zero_grad() と同じ役割
    loss = (w * 3) ** 2
    loss.backward()
    print(f"step {step}: w.grad = {w.grad.item()}")   # 常に 18.0

## 4.3　ネットワークを組み立てる ― nn.Module

In [ ]:
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()          # 基底クラス(nn.Module)側の初期化を先に済ませる。省くと層が登録されない
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 畳み込み：局所特徴を抽出
            nn.BatchNorm2d(32),                          # 正規化：学習を安定させる
            nn.ReLU(),                                    # 活性化：非線形性を与える
            nn.MaxPool2d(2),                              # プーリング：解像度を半分に
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                      # 空間を1×1に集約
        )
        self.classifier = nn.Linear(64, num_classes)      # 全結合層：特徴→クラス

    def forward(self, x):                                 # x: (B, 3, H, W)
        x = self.features(x)                              # → (B, 64, 1, 1)
        x = x.flatten(1)                                  # → (B, 64) 平坦化
        return self.classifier(x)                         # → (B, num_classes)

model = SimpleClassifier(num_classes=9)   # 例：9疾患の分類

In [ ]:
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"学習するパラメータ数: {n_params:,}")     # 例: 学習するパラメータ数: 20,169

## 手を動かす ― フックで「層ごとに形がどう変わるか」を実際に印字する

In [ ]:
import torch, torch.nn as nn

net = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),   # 224 → 112
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 112 → 56
    nn.Conv2d(16, 32, 3, padding=1),nn.ReLU(), nn.MaxPool2d(2),   # 56  → 28
)

# 各層の出力の形を、通過するたびに印字するフックを仕掛ける
for i, layer in enumerate(net):
    layer.register_forward_hook(
        lambda m, inp, out, i=i: print(f"層{i:>2} {m.__class__.__name__:<10} → {tuple(out.shape)}")
    )

x = torch.randn(4, 1, 224, 224)   # (B, C, H, W)：4枚のダミーX線
_ = net(x)

## 4.4　データを供給する ― DatasetとDataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class OCTDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df                    # 症例一覧（パスとラベルの表）
        self.transform = transform      # 前処理・データ拡張

    def __len__(self):
        return len(self.df)             # 総症例数

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row["Raw_Image_Path"])   # 1枚読み込む
        label = row["Disease_Label"]
        if self.transform:
            image = self.transform(image)            # 拡張・正規化・テンソル化
        return image, label                          # (画像, ラベル) を返す

train_loader = DataLoader(
    OCTDataset(train_df, transform=train_tf),
    batch_size=16,        # 一度に処理する枚数
    shuffle=True,         # 毎エポック順序を混ぜる（学習では必須）
    num_workers=4,        # 読み込みを並列化するプロセス数
    pin_memory=True,      # GPU転送を高速化
)

## 手を動かす ― 外部ファイル不要の最小データセットを回す

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    """外部ファイル不要の最小データセット。1件=(ダミー画像, ラベル)"""
    def __init__(self, n):
        self.n = n
    def __len__(self):
        return self.n                    # 総数（__len__ の約束）
    def __getitem__(self, idx):
        image = torch.randn(1, 28, 28)   # (C, H, W) のダミー1枚
        label = idx % 2                  # 0/1 を交互に
        return image, label              # (画像, ラベル) を返す

loader = DataLoader(ToyDataset(10), batch_size=4, shuffle=True)

for images, labels in loader:
    print(images.shape, labels)
# torch.Size([4, 1, 28, 28]) tensor([...])   ← 4枚を束ねてバッチ次元が付いた
# torch.Size([4, 1, 28, 28]) tensor([...])
# torch.Size([2, 1, 28, 28]) tensor([...])   ← 端数の最後は2枚だけ

## 4.5　学習ループの骨格

In [ ]:
model = model.to(device)                              # モデルをGPUへ
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)   # 最適化（つまみの回し方）
criterion = nn.CrossEntropyLoss()                     # 損失（ズレの測り方）

for epoch in range(num_epochs):
    model.train()                                     # 学習モード
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)   # データもGPUへ

        optimizer.zero_grad()          # ① 勾配をリセット
        outputs = model(images)        # ② 順伝播（予測）
        loss = criterion(outputs, labels)  # ③ 誤差を計算
        loss.backward()                # ④ 逆伝播（勾配を計算）
        optimizer.step()               # ⑤ パラメータを更新

    # ここで出るのは「そのエポックの最後の1バッチ」の損失。エポック平均ではない。
    # 平均が見たいなら、内側のループで損失を積算して件数で割る。
    print(f"Epoch {epoch}: 最終バッチの loss = {loss.item():.4f}")

## 4.6　検証と推論 ― train と eval を正しく切り替える

In [ ]:
model.eval()                                # 推論モード（BN/Dropoutの挙動を固定）
correct = 0
with torch.no_grad():                       # 勾配を追跡しない（省メモリ・高速）
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)       # 最も確率の高いクラス
        correct += (preds == labels).sum().item()
accuracy = correct / len(val_loader.dataset)
print(f"検証精度: {accuracy:.3f}")

## 4.7　モデルの保存と読み込み ― 学習の成果を残す

In [ ]:
# 保存：重み（state_dict）だけを書き出す
torch.save(model.state_dict(), "best_model.pth")

# 読み込み：同じ設計のモデルを用意してから、重みを流し込む
model = SimpleClassifier(num_classes=9)          # 設計図を先に作る
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device).eval()                          # 推論の準備完了

In [ ]:
# 学習ループの中で、検証成績が最良を更新したときだけ保存
# （best_score はループに入る前に float("-inf") で初期化しておく）
if val_score > best_score:
    best_score = val_score
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),   # 最適化の内部状態（Adamの慣性など）
        "best_score": best_score,
    }, "checkpoint.pth")

## 4.8　GPUとデバイス管理 ― 大きな医療画像を回すために

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)                    # モデルをGPUへ（1回でよい）
images = images.to(device)                  # データは毎バッチGPUへ送る

## 4.9　混合精度学習（AMP）― 精度を保ったまま倍速・省メモリ

In [ ]:
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")

for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()

    with autocast("cuda"):                    # この中の計算を自動でFP16/FP32に振り分け
        outputs = model(images)
        loss = criterion(outputs, labels)

    scaler.scale(loss).backward()             # 損失を定数倍してから逆伝播（勾配も拡大され消失を防ぐ）
    scaler.step(optimizer)                    # 拡大を戻して安全に更新
    scaler.update()                           # 次回の拡大率を調整

## 4.10　転移学習 ― 少ないデータで戦う定石

In [ ]:
import torch.nn as nn
import torchvision.models as models

# ① ImageNetで学習済みのモデルを出発点として読み込む
model = models.efficientnet_b0(weights="IMAGENET1K_V1")

# ② 出力層（分類器）だけを、自分のタスクのクラス数に付け替える
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)  # 例：9疾患

# ③ 少ない医療データで学習する（次に述べる2つの手法のどちらかを選ぶ）
model = model.to(device)
# あとは通常の学習ループを回すだけ
# ただし入力は3チャネル前提。X線・CT・OCTのような1チャネル画像は
# transforms.Grayscale(num_output_channels=3) で複製してから渡す（入門編の「やってみる」の章、および本書の第12章）